# Gold: one flat table for the cohort agent

A single tall table — one row per figure — holding everything the cohort agent should be
able to answer from.

| | |
| --- | --- |
| **Reads** | `gold_referral_state`, `gold_criteria_hits`, `gold_signal_latency`, `gold_equity_check`, `gold_validation_sensitivity` |
| **Writes** | `gold_cohort_summary` |

## Why a serving table rather than the raw tables

Three reasons, in order of how much they matter.

**The agent could only see three of the nine gold tables.** `gold_referral_state` carries
an `array<string>` column, which the SQL analytics endpoint cannot surface at all, and
several others simply never appeared in the agent's discovery. Rather than keep guessing
at why, this gives it one table that is certain to be readable: strings, doubles and
integers only.

**An agent answering from raw tables has to do arithmetic**, and arithmetic is where it
invents things. Here every figure is precomputed by the pipeline. The agent looks a
number up and reports it; it does not derive one.

**Every row carries its own caveat.** The note travels with the figure, so the agent
cannot report a number while leaving behind the sentence that makes it honest — which is
exactly what happens when the caveat lives only in a prompt.

In [ ]:
PIPELINE_RUN_ID = ""

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (DoubleType, StringType, StructField, StructType)

RUN_ID = PIPELINE_RUN_ID or "local"

state = spark.table("gold_referral_state")
hits = spark.table("gold_criteria_hits")
latency = spark.table("gold_signal_latency")
equity = spark.table("gold_equity_check")
validation = spark.table("gold_validation_sensitivity")
print("gold loaded")

In [ ]:
# ---------------------------------------------------------- assemble the rows
rows = []


def add(metric, dimension, group, value, unit, note):
    rows.append({"metric": metric, "dimension": dimension, "group": group,
                 "value": float(value), "unit": unit, "note": note,
                 "run_id": RUN_ID})


total = state.count()
add("cohort_size", "overall", "all", total, "children",
    "Every child in the synthetic cohort. All data is fabricated.")

for row in state.groupBy("referral_state").count().collect():
    caveat = {
        "indicators_present":
            "Criteria fired on the record. NOT a diagnosis and NOT a referral decision.",
        "no_indicators_recorded":
            "The record was read and nothing fired. This does NOT mean the child has no "
            "indication for genetics -- only that nothing was found in the record.",
        "not_screened":
            "Too little record to read. This is NOT a clear screen; nothing was "
            "assessed.",
    }[row["referral_state"]]
    add("children_by_state", "referral_state", row["referral_state"], row["count"],
        "children", caveat)
    add("share_by_state", "referral_state", row["referral_state"],
        round(100.0 * row["count"] / total, 1), "percent", caveat)

for row in hits.groupBy("tier", "criterion").count().collect():
    add("criterion_fire_count", "criterion", row["criterion"], row["count"], "children",
        f"{row['criterion']} is a {row['tier']} criterion. Its threshold is a "
        f"PLACEHOLDER pending sign-off by the genetics service.")

stats = latency.select(
    F.count("*").alias("n"),
    F.round(F.avg("latency_months"), 1).alias("mean"),
    F.round(F.expr("percentile_approx(latency_months, 0.5)"), 1).alias("median"),
    F.round(F.expr("percentile_approx(latency_months, 0.9)"), 1).alias("p90"),
    F.max("latency_months").alias("max")).collect()[0]

LATENCY_NOTE = ("How long the qualifying evidence has ALREADY been in the record. The "
                "synthetic record contains no referral events, so this says the evidence "
                "has been sufficient since that date. It does NOT say a referral was "
                "missed, late or delayed.")
for label, value in [("median", stats["median"]), ("mean", stats["mean"]),
                     ("p90", stats["p90"]), ("max", stats["max"])]:
    add(f"latency_{label}", "overall", "surfaced children", value, "months",
        LATENCY_NOTE)

over_year = latency.filter(F.col("latency_months") >= 12).count()
add("latency_over_12_months", "overall", "surfaced children", over_year, "children",
    f"{over_year} of {stats['n']} surfaced children have had complete, sufficient "
    f"evidence in the record for a year or more. " + LATENCY_NOTE)

for row in equity.filter("dimension = 'interpreter_required'").collect():
    group = "interpreter needed" if row["group"] == "true" else "no interpreter"
    add("flag_rate", "interpreter_required", group,
        round(100.0 * row["flag_rate"], 1), "percent",
        "Share of screened children who surfaced. Nothing in the criteria reads "
        "language or interpreter need.")

for row in validation.collect():
    group = "interpreter needed" if row["group"] == "true" else "no interpreter"
    add("sensitivity", "interpreter_required", group,
        round(100.0 * row["sensitivity"], 1), "percent",
        "Share of AFFECTED children the screen surfaced, against the synthetic answer "
        "key. Both groups carry the same planted prevalence, so the gap is caused by "
        "what reached the record, not by biology. A real deployment cannot compute "
        "this.")

for row in (latency.join(state.select("patient_id", "interpreter_required"),
                         "patient_id")
            .groupBy("interpreter_required")
            .agg(F.round(F.expr("percentile_approx(latency_months, 0.5)"), 1)
                 .alias("median")).collect()):
    group = "interpreter needed" if row["interpreter_required"] else "no interpreter"
    add("latency_median_by_group", "interpreter_required", group, row["median"],
        "months",
        "SURVIVORSHIP WARNING: this looks better for children needing an interpreter "
        "only because the screen missed the subtler cases entirely. Those children are "
        "absent from this figure and present in the sensitivity gap. Do not report this "
        "as evidence that timing is equitable.")

print(f"rows: {len(rows)}")

In [ ]:
# ------------------------------------------------------------------- persist
schema = StructType([
    StructField("metric", StringType()),
    StructField("dimension", StringType()),
    StructField("group", StringType()),
    StructField("value", DoubleType()),
    StructField("unit", StringType()),
    StructField("note", StringType()),
    StructField("run_id", StringType()),
])
summary = spark.createDataFrame(rows, schema)
summary.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_cohort_summary")

print(f"gold_cohort_summary  {summary.count()} rows")
for row in summary.orderBy("metric", "group").collect():
    print(f"  {row['metric']:26} {row['group']:22} {row['value']:>8.1f} {row['unit']}")

# Only simple types, or the SQL analytics endpoint will not surface it and the agent
# will not see it -- which is the whole reason this table exists.
complex_columns = [f.name for f in summary.schema.fields
                   if f.dataType.typeName() in ("array", "map", "struct")]
if complex_columns:
    raise ValueError(f"complex columns would hide this table from the SQL endpoint: "
                     f"{complex_columns}")
print("\nall columns are simple types; the SQL endpoint can surface this")